## Filter each AQI__2010_2024.csv

In [5]:
#pip install pyarrow

   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   ---------------------------------------  27.3/27.4 MB 157.0 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.4 MB 157.0 MB/s eta 0:00:01
   ---------------------------------------- 27.4/27.4 MB 52.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("../../clean/Yashaswi/DimPollutants/")
OUTPUT_DIR = Path("../../clean/Yashaswi/DimPollutants/Pollutants2015/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

for csv_file in INPUT_DIR.glob("AQI_*_2010_2024.csv"):
    print(f"Filtering {csv_file.name}")

    df = pd.read_csv(csv_file, parse_dates=["Date"])

    df_filtered = df[
        (df["Date"] >= START_DATE) &
        (df["Date"] <= END_DATE)
    ]

    out_file = OUTPUT_DIR / csv_file.name.replace(
        "2010_2024", "2015_2024"
    )

    df_filtered.to_csv(out_file, index=False)

    print(f"✅ Saved {out_file.name}: {len(df_filtered):,} rows")

print("🎯 All pollutant files filtered")

Filtering AQI_CO_2010_2024.csv
✅ Saved AQI_CO_2015_2024.csv: 223,006 rows
Filtering AQI_NO2_2010_2024.csv
✅ Saved AQI_NO2_2015_2024.csv: 676,080 rows
Filtering AQI_O3_2010_2024.csv
✅ Saved AQI_O3_2015_2024.csv: 804,863 rows
Filtering AQI_PM10_2010_2024.csv
✅ Saved AQI_PM10_2015_2024.csv: 138,338 rows
Filtering AQI_PM2.5_2010_2024.csv
✅ Saved AQI_PM2.5_2015_2024.csv: 733,696 rows
Filtering AQI_SO2_2010_2024.csv
✅ Saved AQI_SO2_2015_2024.csv: 507,332 rows
🎯 All pollutant files filtered


In [11]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("../../clean/Yashaswi/DimPollutants/Pollutants2015/")
OUTPUT_DIR = Path("../../clean/Yashaswi/DimPollutants/Pollutants2015_Parquet/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DTYPES = {
    "NAPS ID": "int32",
    "AQI": "float32",          # AQI does NOT need float64
    "Pollutant": "category"   # HUGE compression win
}

for csv_file in INPUT_DIR.glob("AQI_*_2015_2024.csv"):
    print(f"Optimizing {csv_file.name}")

    df = pd.read_csv(
        csv_file,
        dtype=DTYPES,
        parse_dates=["Date"]
    )

    out_file = OUTPUT_DIR / csv_file.with_suffix(".parquet").name
    df.to_parquet(
        out_file,
        compression="snappy",
        index=False
    )

    print(f"✅ Saved {out_file.name}")

print("🎯 All files converted to Parquet")


Optimizing AQI_CO_2015_2024.csv
✅ Saved AQI_CO_2015_2024.parquet
Optimizing AQI_NO2_2015_2024.csv
✅ Saved AQI_NO2_2015_2024.parquet
Optimizing AQI_O3_2015_2024.csv
✅ Saved AQI_O3_2015_2024.parquet
Optimizing AQI_PM10_2015_2024.csv
✅ Saved AQI_PM10_2015_2024.parquet
Optimizing AQI_PM2.5_2015_2024.csv
✅ Saved AQI_PM2.5_2015_2024.parquet
Optimizing AQI_SO2_2015_2024.csv
✅ Saved AQI_SO2_2015_2024.parquet
🎯 All files converted to Parquet


## ✅ Script 2: Filter consolidated Station wise daily AQI file

In [12]:
import pandas as pd

# Read original consolidated daily AQI file
df = pd.read_csv(
    "AQI_2010_2024.csv",
    parse_dates=["Date"]
)

# Filter date range
df = df[
    (df["Date"] >= "2015-01-01") &
    (df["Date"] <= "2024-12-31")
]

# ✅ Optimize dtypes for Power BI
df["NAPS ID"] = df["NAPS ID"].astype("int32")
df["AQI"] = df["AQI"].astype("float32")

# If present (very likely)
if "Dominant_Pollutant" in df.columns:
    df["Dominant_Pollutant"] = df["Dominant_Pollutant"].astype("category")

# Save as Parquet (Power BI–optimized)
df.to_parquet(
    "AQI_2015_2024.parquet",
    compression="snappy",
    index=False
)

print(f"✅ Rows after filtering: {len(df):,}")

✅ Rows after filtering: 981,507


In [13]:
import pandas as pd

# Load AQI data (Parquet)
aqi = pd.read_parquet("AQI_2015_2024.parquet")

# Load station master list
stations = pd.read_csv("../../raw/Yashaswi/AQI/NAPS Stations.csv")

In [14]:
null_naps = aqi["NAPS ID"].isna().sum()

print(f"🚨 AQI rows with NULL NAPS ID: {null_naps}")

🚨 AQI rows with NULL NAPS ID: 0


In [15]:
aqi[aqi["NAPS ID"].isna()].head()

,NAPS ID,Date,AQI,Dominant_Pollutant


In [16]:
missing_in_stations = set(aqi["NAPS ID"]) - set(stations["NAPS_ID"])

print(f"⚠ AQI NAPS IDs not found in station file: {len(missing_in_stations)}")

⚠ AQI NAPS IDs not found in station file: 313


In [17]:
print(stations.columns)

Index(['NAPS_ID', 'Station_Name', 'City', 'Province/Territory_EN', 'Latitude',
       'Longitude', 'Elevation_m', 'Start_Date_Station_yyyy_mm_dd'],
      dtype='str')


In [18]:
aqi["NAPS_ID"] = aqi["NAPS ID"].astype("int32")
stations["NAPS_ID"] = stations["NAPS_ID"].astype("int32")

In [19]:
missing_in_stations = set(aqi["NAPS_ID"]) - set(stations["NAPS_ID"])
print(len(missing_in_stations))


313


In [20]:
missing_rows = aqi[aqi["NAPS_ID"].isin(missing_in_stations)]

print(f"AQI rows affected: {len(missing_rows):,}")


AQI rows affected: 812,856


In [23]:
import pandas as pd

stations_naps = pd.read_csv("../../clean/Yashaswi/DimPollutants/StationsNAPS.csv")

print(stations_naps.columns)

Index(['*Stations2024', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29',
       'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33',
       'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37',
       'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41',
       'Unnamed: 42'],
      dtype='str')


In [25]:
stations_naps.head(10)

,*Stations2024,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42
0,*Definitions_Définitions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,*DataSources_SourcesdesDonnées,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,**********Stations2024**********,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NAPS_ID,Station_Name,Status,Location_Address,City,Province/Territory_EN,Province/territoire_FR,Postal_Code,Timezone_UTC,Latitude,...,Urbanization,Neighbourhood,Land_Use,Scale,PC,CD,CSD,CMA/CA,AQMS_Airzone,Core_Site
4,Identifiant_SNPA,Nom_de_la_station,Statut,Adresse_de_l'emplacement,Ville,Province/Territory_EN,Province/territoire_FR,Code_postal,Fuseau_horaire_TUC,Latitude,...,Urbanisation,Quartier,Utilisation_du_sol,Échelle,CP,DR,SDR,RMR/AR,SGQA_zone_atmosphérique,Site_principal
5,10101,DUCKWORTH & ORDINANCE,0,DUCKWORTH & ORDINANCE,ST. JOHN'S,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1C 1E4,-3.5,47.56806,...,LU,P4,R,N,0792. St. John's,01. Division No. 1,"519. St. John's, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,NaN
6,10102,WATER STREET POST OFFICE,1,354 WATER STREET,ST. JOHN'S,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1C 1C4,-3.5,47.56038,...,LU,P4,C,N,0792. St. John's,01. Division No. 1,"519. St. John's, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,2
7,10301,CREDIT UNION,0,BROOK STREET,CORNER BROOK,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A2H 2T7,-3.5,48.949479,...,SU,P3,I,N,0204. Corner Brook,05. Division No. 5,"018. Corner Brook, CY","015. Corner Brook, CA/AR",Newfoundland and Labrador - Newfoundland,NaN
8,10401,MOUNT PEARL,1,OLD PLACENTIA ROAD,MOUNT PEARL,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1N 4P4,-3.5,47.50513,...,LU,P3,R,N,0792. St. John's,01. Division No. 1,"542. Mount Pearl, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,NaN
9,10501,ABITIBI SITE,1,SCOTT AVENUE,GRAND FALLS - WINDSOR,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A2A 2K1,-3.5,48.92696,...,SU,P3,I,N,0332. Grand Falls-Windsor,06. Division No. 6,"017. Grand Falls-Windsor, T","010. Grand Falls-Windsor, CA/AR",Newfoundland and Labrador - Newfoundland,NaN


In [26]:
import pandas as pd

# Read raw file without headers
raw = pd.read_csv("../../clean/Yashaswi/DimPollutants/StationsNAPS.csv", header=None)

# Find the row where the real header starts
header_row = raw.index[raw[0] == "NAPS_ID"][0]

# Re-read using that row as header
stations = pd.read_csv(
    "../../clean/Yashaswi/DimPollutants/StationsNAPS.csv",
    header=header_row
)

stations.head()

,NAPS_ID,Station_Name,Status,Location_Address,City,Province/Territory_EN,Province/territoire_FR,Postal_Code,Timezone_UTC,Latitude,...,Urbanization,Neighbourhood,Land_Use,Scale,PC,CD,CSD,CMA/CA,AQMS_Airzone,Core_Site
0,Identifiant_SNPA,Nom_de_la_station,Statut,Adresse_de_l'emplacement,Ville,Province/Territory_EN,Province/territoire_FR,Code_postal,Fuseau_horaire_TUC,Latitude,...,Urbanisation,Quartier,Utilisation_du_sol,Échelle,CP,DR,SDR,RMR/AR,SGQA_zone_atmosphérique,Site_principal
1,10101,DUCKWORTH & ORDINANCE,0,DUCKWORTH & ORDINANCE,ST. JOHN'S,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1C 1E4,-3.5,47.56806,...,LU,P4,R,N,0792. St. John's,01. Division No. 1,"519. St. John's, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,NaN
2,10102,WATER STREET POST OFFICE,1,354 WATER STREET,ST. JOHN'S,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1C 1C4,-3.5,47.56038,...,LU,P4,C,N,0792. St. John's,01. Division No. 1,"519. St. John's, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,2
3,10301,CREDIT UNION,0,BROOK STREET,CORNER BROOK,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A2H 2T7,-3.5,48.949479,...,SU,P3,I,N,0204. Corner Brook,05. Division No. 5,"018. Corner Brook, CY","015. Corner Brook, CA/AR",Newfoundland and Labrador - Newfoundland,NaN
4,10401,MOUNT PEARL,1,OLD PLACENTIA ROAD,MOUNT PEARL,NEWFOUNDLAND AND LABRADOR,TERRE-NEUVE-ET-LABRADOR,A1N 4P4,-3.5,47.50513,...,LU,P3,R,N,0792. St. John's,01. Division No. 1,"542. Mount Pearl, CY","001. St. John's, CMA/RMR",Newfoundland and Labrador - Newfoundland,NaN


In [2]:
stations.columns

Index(['NAPS_ID', 'Station_Name', 'Status', 'Location_Address', 'City',
       'Province/Territory_EN', 'Province/territoire_FR', 'Postal_Code',
       'Timezone_UTC', 'Latitude', 'Longitude', 'Elevation_m', 'Start_Date',
       'End_Date', 'Combined_Stations', 'Inlet_Height_m', 'Network', 'SO2',
       'CO', 'NO2', 'NO', 'NOX', 'O3', 'PM2.5_Continuous', 'PM10_Continuous',
       'PM2.5_RM', 'PM10_Integrated', 'PM2.5-10', 'PM2.5_Speciation', 'VOC',
       'Carbonyl', 'PAC', 'Site_Type', 'Urbanization', 'Neighbourhood',
       'Land_Use', 'Scale', 'PC', 'CD', 'CSD', 'CMA/CA', 'AQMS_Airzone',
       'Core_Site'],
      dtype='str')

In [27]:
stations[stations["NAPS_ID"].isna()]

,NAPS_ID,Station_Name,Status,Location_Address,City,Province/Territory_EN,Province/territoire_FR,Postal_Code,Timezone_UTC,Latitude,...,Urbanization,Neighbourhood,Land_Use,Scale,PC,CD,CSD,CMA/CA,AQMS_Airzone,Core_Site
655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
716,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
730,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
734,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
stations = stations.dropna(subset=["NAPS_ID"])

In [29]:
stations["NAPS_ID"] = pd.to_numeric(
    stations["NAPS_ID"],
    errors="coerce"
)

In [30]:
stations = stations.dropna(subset=["NAPS_ID"])


In [31]:
stations["NAPS_ID"] = stations["NAPS_ID"].astype("int32")

In [32]:
stations["NAPS_ID"].isna().sum()

np.int64(0)

In [33]:
aqi["NAPS_ID"] = aqi["NAPS_ID"].astype("int32")

missing_in_stations = set(aqi["NAPS_ID"]) - set(stations["NAPS_ID"])
print("Missing NAPS IDs:", len(missing_in_stations))

Missing NAPS IDs: 0


In [34]:
stations.columns

Index(['NAPS_ID', 'Station_Name', 'Status', 'Location_Address', 'City',
       'Province/Territory_EN', 'Province/territoire_FR', 'Postal_Code',
       'Timezone_UTC', 'Latitude', 'Longitude', 'Elevation_m', 'Start_Date',
       'End_Date', 'Combined_Stations', 'Inlet_Height_m', 'Network', 'SO2',
       'CO', 'NO2', 'NO', 'NOX', 'O3', 'PM2.5_Continuous', 'PM10_Continuous',
       'PM2.5_RM', 'PM10_Integrated', 'PM2.5-10', 'PM2.5_Speciation', 'VOC',
       'Carbonyl', 'PAC', 'Site_Type', 'Urbanization', 'Neighbourhood',
       'Land_Use', 'Scale', 'PC', 'CD', 'CSD', 'CMA/CA', 'AQMS_Airzone',
       'Core_Site'],
      dtype='str')

In [35]:
dim_station = stations[
    [
        "NAPS_ID",
        "Station_Name",
        "City",
        "Province/Territory_EN",
        "Latitude",
        "Longitude",
        "Status",
        "Start_Date",
        "End_Date"
    ]
]


In [36]:
province_map = {
    "NEWFOUNDLAND AND LABRADOR": "NL",
    "PRINCE EDWARD ISLAND": "PE",
    "NOVA SCOTIA": "NS",
    "NEW BRUNSWICK": "NB",
    "QUEBEC": "QC",
    "ONTARIO": "ON",
    "MANITOBA": "MB",
    "SASKATCHEWAN": "SK",
    "ALBERTA": "AB",
    "BRITISH COLUMBIA": "BC",
    "YUKON": "YT",
    "NORTHWEST TERRITORIES": "NT",
    "NUNAVUT": "NU"
}

In [37]:
dim_station["Province_Code"] = (
    dim_station["Province/Territory_EN"]
    .str.upper()
    .map(province_map)
)

In [38]:
dim_station[
    dim_station["Province_Code"].isna()
][["Province/Territory_EN"]].drop_duplicates()


,Province/Territory_EN


In [39]:
dim_station = dim_station[
    [
        "NAPS_ID",
        "Station_Name",
        "City",
        "Province/Territory_EN",
        "Province_Code",
        "Latitude",
        "Longitude",
        "Status",
        "Start_Date",
        "End_Date"
    ]
] = dim_station[
    [
        "NAPS_ID",
        "Station_Name",
        "City",
        "Province/Territory_EN",
        "Province_Code",
        "Latitude",
        "Longitude",
        "Status",
        "Start_Date",
        "End_Date"
    ]
]

In [41]:
import pandas as pd

dim_station["Start_Date"] = pd.to_datetime(dim_station["Start_Date"], errors="coerce")
dim_station["End_Date"] = pd.to_datetime(dim_station["End_Date"], errors="coerce")

In [42]:
cutoff_date = pd.Timestamp("2014-01-01")

stations_closed_before_2014 = dim_station[
    (dim_station["End_Date"].notna()) &
    (dim_station["End_Date"] < cutoff_date)
]

In [43]:
stations_closed_before_2014[
    ["NAPS_ID", "Station_Name", "City", "Province_Code", "Start_Date", "End_Date", "Status"]
].sort_values("End_Date")

,NAPS_ID,Station_Name,City,Province_Code,Start_Date,End_Date,Status
43,40201,SAINT JOHN - 110 CHARLOTTE ST.,SAINT JOHN,NB,1974-01-23,1974-02-06,0
374,90220,CALGARY - 26 AVE. & 4 ST. N.W.,CALGARY,AB,1974-01-01,1974-06-30,0
451,100105,VANCOUVER - 739 WEST HASTINGS ST.,METRO VANCOUVER - VANCOUVER,BC,1974-01-01,1975-01-31,0
328,70103,HARTFORD & MAIN,WINNIPEG,MB,1974-01-04,1976-01-15,0
101,50201,GATINEAU - EDDY,GATINEAU,QC,1974-01-01,1976-06-04,0
...,...,...,...,...,...,...,...
505,100317,VICTORIA - JAMES BAY DANIELS ELECTRONICS,VICTORIA,BC,2011-04-12,2013-12-12,0
610,106700,CROFTON - ESCARPMENT WAY,CROFTON,BC,2009-01-01,2013-12-31,0
192,60303,KINGSTON - 2,KINGSTON,ON,2007-01-01,2013-12-31,0
36,30801,YARMOUTH - DAYTON,YARMOUTH,NS,1994-01-05,2013-12-31,0


In [44]:
import pandas as pd

dim_station["Start_Date"] = pd.to_datetime(dim_station["Start_Date"], errors="coerce")
dim_station["End_Date"] = pd.to_datetime(dim_station["End_Date"], errors="coerce")

In [48]:
cutoff_date = pd.Timestamp("2015-01-01")

dim_station = dim_station[
    (dim_station["End_Date"].isna()) |
    (dim_station["End_Date"] >= cutoff_date)
]

In [49]:
print("Remaining stations:", len(dim_station))
print(
    "Any stations still closed before 2015:",
    ((dim_station["End_Date"].notna()) & (dim_station["End_Date"] < cutoff_date)).any()
)

Remaining stations: 370
Any stations still closed before 2015: False


In [50]:


dim_station.to_csv(
    "Dim_Station_NAPS.csv",
    index=False
)